# Тут будет прмиер кода на питоне

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

# 1. ЗАГРУЗКА ДАННЫХ ИЗ ФАЙЛА ПРОЕКТА
file_path = 'queries_log.jsonl'

if os.path.exists(file_path):
    # Читаем jsonl (каждая строка — отдельный объект)
    df = pd.read_json(file_path, lines=True)
    
    # Приведение типов и извлечение временных признаков
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour'] = df['timestamp'].dt.hour
    
    # Расчет метрик: количество токенов (слов) в запросе
    df['tokens'] = df['query'].str.strip().str.split().str.len()
    
    print(f"Успешно загружено записей: {len(df)}")
else:
    print(f"Ошибка: Файл {file_path} не найден в директории проекта.")

# Цветовая палитра Revolut
REV_BLUE = '#0075FF'
REV_BLACK = '#191919'
REV_RED = '#FF3B30'
REV_GREY = '#8E8E93'

# --- 1. АНАЛИЗ СЛОЖНОСТИ ЗАПРОСОВ (ТОКЕНЫ) ---
avg_tokens = df.groupby('category')['tokens'].mean().reset_index().sort_values('tokens', ascending=False)

fig_tokens = go.Figure(go.Bar(
    x=avg_tokens['category'], y=avg_tokens['tokens'],
    marker_color=REV_BLUE,
    text=avg_tokens['tokens'].round(2),
    textposition='outside',
    cliponaxis=False
))
fig_tokens.update_layout(
    title="<b>СРЕДНЯЯ ДЛИНА ЗАПРОСА ПО КАТЕГОРИЯМ (ТОКЕНЫ)</b>",
    template="plotly_white",
    yaxis=dict(title="Среднее кол-во слов", gridcolor='#F2F2F7'),
    xaxis_title=""
)
fig_tokens.show()

# --- 2. ВРЕМЕННАЯ АКТИВНОСТЬ (ИНТЕНСИВНОСТЬ ПО ЧАСАМ) ---
time_dist = df.groupby(['hour', 'category']).size().reset_index(name='count')

fig_time = px.line(
    time_dist, x='hour', y='count', color='category',
    line_shape='spline',
    title="<b>РАСПРЕДЕЛЕНИЕ ПОИСКОВОЙ АКТИВНОСТИ</b>",
    color_discrete_map={'IT': REV_BLUE, 'Science': REV_BLACK, 'Art': REV_RED, 'Culture': REV_GREY}
)
fig_time.update_layout(
    template="plotly_white",
    xaxis=dict(title="Час суток", dtick=1, showgrid=False),
    yaxis=dict(title="Кол-во запросов", gridcolor='#F2F2F7'),
    legend_title="Категория"
)
fig_time.show()

# --- 3. ЯЗЫКОВОЙ АНАЛИЗ: СООТНОШЕНИЕ RU/EN ПО КАТЕГОРИЯМ ---
lang_stats = df.groupby(['category', 'language']).size().reset_index(name='count')

fig_lang = px.bar(
    lang_stats, x='category', y='count', color='language',
    title="<b>ЯЗЫКОВОЙ ИНДЕКС: РАСПРЕДЕЛЕНИЕ RU VS EN</b>",
    barmode='stack',
    color_discrete_map={'en': REV_BLUE, 'ru': REV_BLACK}
)
fig_lang.update_layout(
    template="plotly_white",
    xaxis_title="",
    yaxis_title="Кол-во запросов",
    legend_title="Язык"
)
fig_lang.show()

Успешно загружено записей: 116
